In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../dataset/bengaluru_house_prices.csv")

df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [3]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Rows : 13320
Columns : 9


In [4]:
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [5]:
df = df.drop("society", axis=1)

df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200,2.0,1.0,51.00


### Observation

The `society` column was removed because it contains a large number of missing values and contributes little to house price prediction.

In [6]:
df = df.dropna()

df.isnull().sum()

area_type       0
availability    0
location        0
size            0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64

### Observation

Rows containing missing values were removed to obtain a clean dataset for machine learning.

In [7]:
df["BHK"] = df["size"].str.extract("(\d+)")

df["BHK"] = df["BHK"].astype(int)

df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price,BHK
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056,2.0,1.0,39.07,2
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600,5.0,3.0,120.00,4
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440,2.0,3.0,62.00,3
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521,3.0,1.0,95.00,3
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200,2.0,1.0,51.00,2


### Observation

The BHK feature was extracted from the `size` column and converted into a numerical feature for model training.

In [8]:
def convert_sqft_to_num(x):
    tokens = str(x).split('-')

    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2

    try:
        return float(x)

    except:
        return None

In [9]:
df["total_sqft"] = df["total_sqft"].apply(convert_sqft_to_num)

df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price,BHK
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056.0,2.0,1.0,39.07,2
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600.0,5.0,3.0,120.00,4
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440.0,2.0,3.0,62.00,3
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521.0,3.0,1.0,95.00,3
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200.0,2.0,1.0,51.00,2


### Observation

The `total_sqft` column contained values such as ranges (e.g., 2100-2850). These were converted into their average values to create a numerical feature suitable for machine learning.

In [10]:
df = df.dropna()

df.shape

(12668, 9)

In [11]:
df["price_per_sqft"] = (df["price"] * 100000) / df["total_sqft"]

df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price,BHK,price_per_sqft
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056.0,2.0,1.0,39.07,2,3699.810606
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600.0,5.0,3.0,120.00,4,4615.384615
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440.0,2.0,3.0,62.00,3,4305.555556
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521.0,3.0,1.0,95.00,3,6245.890861
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200.0,2.0,1.0,51.00,2,4250.000000


### Observation

A new feature named `price_per_sqft` was created. This helps identify unusual property prices and improves outlier detection.

In [12]:
df = df[df["total_sqft"] / df["BHK"] >= 300]

df.shape

(12013, 10)

### Observation

Properties with less than 300 square feet per BHK were treated as unrealistic and removed from the dataset.

In [13]:
df = df.drop(
    ["size", "availability"],
    axis=1
)

df.head()

,area_type,location,total_sqft,bath,balcony,price,BHK,price_per_sqft
0,Super built-up Area,Electronic City Phase II,1056.0,2.0,1.0,39.07,2,3699.810606
1,Plot Area,Chikka Tirupathi,2600.0,5.0,3.0,120.00,4,4615.384615
2,Built-up Area,Uttarahalli,1440.0,2.0,3.0,62.00,3,4305.555556
3,Super built-up Area,Lingadheeranahalli,1521.0,3.0,1.0,95.00,3,6245.890861
4,Super built-up Area,Kothanur,1200.0,2.0,1.0,51.00,2,4250.000000


### Observation

Columns that were no longer required after feature engineering were removed to simplify the dataset.

In [14]:
df.to_csv("../dataset/clean_house_data.csv", index=False)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


# Feature Engineering Summary

### Completed Tasks

- Removed unnecessary columns.
- Handled missing values.
- Converted `total_sqft` to numerical values.
- Extracted BHK from the `size` column.
- Created `price_per_sqft`.
- Removed unrealistic property records.
- Saved the cleaned dataset for model training.

### Conclusion

The dataset is now cleaned, transformed, and ready for machine learning model development.